# FILA-Net — end-to-end competition pipeline

This notebook illustrates the complete reproducible path required for the
Solar Filament Segmentation Challenge 2026:

1. environment + dataset audit;
2. leakage-safe grouped folds;
3. five-fold training;
4. OOF prediction caching;
5. official-protocol PQ evaluation and diagnostics;
6. nested post-processing calibration;
7. ablation campaign;
8. five-fold ensemble inference;
9. submission structural/RLE verification.

The competition data are not redistributed. Attach/download the official Kaggle
competition dataset and set `data.root` in `configs/filanet.yaml`.


In [ ]:
from pathlib import Path
import json, os, platform, subprocess, sys
import yaml

REPO = Path.cwd()
CONFIG = REPO / "configs/filanet.yaml"

print("Python:", sys.version)
print("Platform:", platform.platform())
print("Repository:", REPO)
print("Config exists:", CONFIG.exists())
print(yaml.safe_load(CONFIG.read_text())["data"])


## 1. Audit the official training data

Strict mode checks the expected competition split structure (707 physical train
images represented by 1,154 independent annotation records and 8,199 filament
instances), image dimensions, missing files, and fold isolation.


In [ ]:
subprocess.run([
    sys.executable, "scripts/audit_data.py",
    "--config", str(CONFIG),
    "--strict",
    "--output", "artifacts/data_audit.json",
], check=True)


## 2. Train all five folds

Training is disabled by default so opening the notebook does not accidentally
start a long GPU job. Set `RUN_TRAIN=1` in the notebook environment to train.
Every annotator record of the same physical JPEG remains in the same fold.


In [ ]:
RUN_TRAIN = os.environ.get("RUN_TRAIN", "0") == "1"
checkpoint_dir = Path("checkpoints")
checkpoint_dir.mkdir(exist_ok=True)

if RUN_TRAIN:
    for fold in range(5):
        subprocess.run([
            sys.executable, "scripts/train.py",
            "--config", str(CONFIG),
            "--fold", str(fold),
            "--output", str(checkpoint_dir),
        ], check=True)
else:
    print("Training skipped. Set RUN_TRAIN=1 to execute all folds.")


## 3. Cache leakage-safe OOF predictions

Predictions are cached once per physical image, even when the image has two or
three independent annotation records. This makes post-processing sweeps much
faster and ensures every GT record is scored against the same model output.


In [ ]:
weights_pattern = "checkpoints/fold{fold}_best.pt"
have_all_weights = all(Path(weights_pattern.format(fold=f)).exists() for f in range(5))

if have_all_weights:
    subprocess.run([
        sys.executable, "scripts/cache_oof.py",
        "--config", str(CONFIG),
        "--weights-pattern", weights_pattern,
        "--cache-dir", "artifacts/oof_cache",
    ], check=True)
else:
    print("OOF cache skipped: five fold checkpoints are not present yet.")


## 4. OOF PQ + morphology diagnostics

The evaluator reports both macro PQ (mean over independent annotation records)
and micro PQ, TP/FP/FN, matched IoU/Dice distributions, fragmentation
(one-to-many) and merging (many-to-one), plus a bootstrap 95% interval.


In [ ]:
if Path("artifacts/oof_cache").exists():
    subprocess.run([
        sys.executable, "scripts/evaluate_oof.py",
        "--config", str(CONFIG),
        "--cache-dir", "artifacts/oof_cache",
        "--output", "artifacts/oof_metrics.json",
        "--records-csv", "artifacts/oof_records.csv",
    ], check=True)
    metrics = json.loads(Path("artifacts/oof_metrics.json").read_text())
    display(metrics)


## 5. Nested post-processing optimization

Candidate thresholds are selected without using the held-out outer fold. The
reported nested score is the guard against post-processing selection bias.
A separate full-OOF best configuration is written for final retraining/inference.


In [ ]:
if Path("artifacts/oof_cache").exists():
    subprocess.run([
        sys.executable, "scripts/optimize_postprocess.py",
        "--config", str(CONFIG),
        "--search-config", "configs/postprocess_search.yaml",
        "--cache-dir", "artifacts/oof_cache",
        "--output", "artifacts/postprocess_search.json",
        "--write-config", "configs/filanet_tuned.yaml",
    ], check=True)
    search = json.loads(Path("artifacts/postprocess_search.json").read_text())
    display(search["nested_selection"])
    display(search["final_params"])


## 6. Single-variable ablation campaign

Generate the complete baseline → +topology → +orientation → +graph bridge →
+TTA experiment manifest. Pass `--execute` only on a machine with the official
data and sufficient GPU time.


In [ ]:
subprocess.run([
    sys.executable, "scripts/run_ablation.py",
    "--config", str(CONFIG),
    "--output-dir", "artifacts/ablations",
], check=True)
print(Path("artifacts/ablations/manifest.json").read_text()[:4000])


## 7. Five-fold ensemble inference and hard submission gate

Use the tuned config only after it has been produced by OOF calibration.
`infer.py` automatically decodes every produced RLE, checks image dimensions,
empty masks, duplicate IDs, unknown image IDs and overlapping predicted
instances, and writes a SHA-256 verification report.


In [ ]:
FINAL_CONFIG = Path("configs/filanet_tuned.yaml")
if not FINAL_CONFIG.exists():
    FINAL_CONFIG = CONFIG

weights = [Path(f"checkpoints/fold{f}_best.pt") for f in range(5)]
if all(p.exists() for p in weights):
    subprocess.run([
        sys.executable, "scripts/infer.py",
        "--config", str(FINAL_CONFIG),
        "--weights", *map(str, weights),
        "--output", "submission.csv",
    ], check=True)
    display(json.loads(Path("submission.verification.json").read_text()))
else:
    print("Final inference skipped: five checkpoints are not present yet.")


## 8. Final review artifacts

Before final submission, retain:

- `artifacts/data_audit.json`
- five best fold checkpoints and their hashes
- `artifacts/oof_metrics.json` + `oof_records.csv`
- `artifacts/postprocess_search.json`
- `artifacts/ablations/*/metrics.json`
- `submission.csv` + `submission.verification.json`
- the four-page technical report built from measured results only

Do not report unmeasured improvements as competition results.
